In [2]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from tqdm import tqdm
import os

# Set working directory if needed
os.chdir("../data")

# Load the sampled comments CSV
df = pd.read_csv("sampled_for_vader.csv")

# Initialize VADER sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

# Add progress bar
tqdm.pandas()

# Function to get all four VADER scores
def get_all_sentiment_scores(text):
    if isinstance(text, str):
        return analyzer.polarity_scores(text)
    return {'neg': 0.0, 'neu': 0.0, 'pos': 0.0, 'compound': 0.0}

# Apply VADER scoring to each comment
scores = df['comment'].progress_apply(get_all_sentiment_scores)

# Convert the dictionary results into separate columns
scores_df = pd.json_normalize(scores)

# Combine original data with scores
df = pd.concat([df, scores_df], axis=1)

# Create a label column (positive/neutral/negative)
def classify_sentiment(score):
    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

df['vader_sentiment'] = df['compound'].apply(classify_sentiment)

# Save the final output
df.to_csv("vader_labeled_comments_full.csv", index=False)

print("✅ Done! All VADER scores saved in 'vader_labeled_comments_full.csv'.")

100%|██████████████████████████████████| 80000/80000 [00:02<00:00, 36959.60it/s]


✅ Done! All VADER scores saved in 'vader_labeled_comments_full.csv'.


In [4]:
df.head()